# 02 - Delta Live Tables Pipeline

This notebook transforms raw JSON data into structured tables using medallion architecture:

- Bronze → Raw ingestion
- Silver → Cleaned structured data
- Gold → Aggregated business insights

The raw data contains nested JSON which needs to be flattened.

## Step 1 - Read Raw Data from Volume

We read JSON files stored in the Unity Catalog Volume.

In [0]:
raw_path = "/Volumes/databricks_workspace/stock_dlt/raw_data"

df_raw = spark.read.json(raw_path)

display(df_raw)

Meta Data Time Series (Daily) List(Daily Prices (open, high, low, close) and Volumes, IBM, 2026-04-23, Compact, US/Eastern) List(List(304.0600, 309.1800, 303.6000, 308.5800, 1689031), List(306.5050, 307.1200, 302.8000, 305.6700, 3166555), List(307.0000, 310.4675, 301.5700, 301.7800, 4261100), List(302.8800, 303.9700, 298.9050, 302.6200, 3953390), List(302.8750, 309.6100, 302.5400, 307.9900, 2962463), List(308.5900, 311.8300, 307.1800, 307.9400, 2344667), List(309.6200, 315.3454, 307.9500, 309.1800, 3615794), List(309.6300, 313.9700, 308.7500, 310.4800, 2914275), List(310.2300, 314.6900, 306.6513, 312.6700, 3411524), List(312.0000, 313.4400, 308.4000, 310.7400, 2755749), List(310.5700, 311.0500, 303.3300, 309.2400, 2953374), List(308.9850, 311.3629, 306.3500, 308.6600, 3566732), List(307.3150, 307.3800, 300.4200, 303.1800, 3366878), List(303.7900, 306.2500, 303.0800, 303.3200, 3130954), List(303.1500, 304.4500, 295.7000, 300.4500, 5411669), List(300.3500, 306.8600, 299.1000, 300.9800, 11031341), List(301.0300, 303.1800, 298.3200, 302.7900, 2612100), List(301.3400, 305.1300, 300.6500, 303.7800, 2923037), List(303.7600, 305.1500, 302.3000, 304.5600, 1210642), List(304.6900, 305.7500, 303.6650, 305.0900, 2814732), List(304.6500, 310.0000, 303.7500, 305.7400, 4664711), List(306.1500, 306.2350, 302.0000, 302.0500, 1883651), List(301.7600, 301.8500, 295.8700, 296.2100, 3430133), List(297.5600, 297.5699, 289.0000, 291.5000, 4662804), List(295.7700, 299.1900, 294.2500, 294.9700, 4189960), List(295.0000, 303.0400, 294.4200, 302.4700, 4147315), List(302.5000, 304.3100, 296.3450, 296.7300, 2833274), List(295.0000, 303.6700, 295.0000, 302.7200, 3343273), List(302.6100, 307.0000, 302.0000, 304.2200, 2718828), List(302.6200, 312.3300, 299.9600, 312.1800, 3895197), List(311.6000, 312.8100, 301.8700, 303.1600, 4507760), List(303.5000, 309.1900, 301.5000, 309.0300, 3779045), List(309.0000, 311.8800, 297.0400, 297.9500, 4932480), List(301.0000, 307.4500, 300.7800, 305.6700, 6199635), List(301.3500, 301.6000, 290.1600, 291.3500, 7211706), List(292.7600, 297.6700, 292.5100, 297.5400, 5185023), List(299.4200, 300.9300, 293.5300, 294.6700, 3670152), List(294.0700, 294.3350, 289.7900, 292.4400, 3298424), List(293.1600, 296.8150, 293.1400, 296.3300, 3726890), List(297.1600, 297.3300, 293.2700, 293.8600, 2944722), List(294.1700, 295.9500, 291.2601, 294.1600, 5790347), List(317.8600, 319.9000, 303.4700, 309.2400, 10124929), List(307.6000, 307.7830, 299.7300, 306.7000, 5940669), List(307.5100, 316.6400, 306.4100, 314.7300, 4581189), List(312.4000, 312.9750, 283.8500, 294.3100, 11296020), List(291.4100, 291.4100, 278.9600, 289.0500, 8708033), List(286.1000, 291.8100, 285.1000, 289.8900, 5532797), List(292.5000, 299.8900, 290.6570, 298.9300, 3744313), List(295.9100, 297.7200, 291.4200, 296.3400, 4627815), List(294.9900, 297.6100, 290.3300, 291.7600, 3837343), List(292.3400, 293.5000, 272.3601, 272.8100, 7628244), List(270.3000, 271.3000, 257.2200, 259.5200, 12565380), List(260.0000, 264.6600, 256.6400, 262.3800, 6842620), List(259.2000, 260.7000, 254.6500, 258.3100, 4929733), List(258.6400, 261.1100, 256.2500, 260.7900, 3949229), List(256.0000, 258.2800, 253.5110, 256.2800, 4948700), List(255.1950, 259.0400, 253.8000, 257.1600, 4708550), List(254.3700, 255.1900, 220.7200, 223.3500, 19522881), List(227.8000, 236.5937, 223.6300, 229.3200, 13379817), List(233.2200, 239.5500, 231.2200, 237.5400, 8569713), List(239.7100, 247.4899, 238.9500, 242.0100, 7343055), List(238.0700, 240.2100, 234.5650, 240.2100, 6642222), List(235.7000, 240.7800, 233.7800, 239.3700, 6220287), List(236.3500, 246.0900, 234.2900, 245.2800, 6960369), List(245.7450, 250.8500, 244.9550, 250.0600, 6084995), List(249.3200, 260.3800, 249.0000, 256.5500, 9899962), List(256.4400, 259.3999, 252.2100, 258.8500, 6234402), List(255.3800, 258.0800, 251.5710, 253.3300, 6126567), List(253.2600, 253.4400, 246.5500, 250.2000, 4937960), List(250.0050, 253.7150, 247.2000, 248.8700, 4012239), List(247.1000, 

## Step 2 - Inspect Schema

We verify nested structure before transformation.

In [0]:
df_raw.printSchema()

root
 |-- Meta Data: struct (nullable = true)
 |    |-- 1. Information: string (nullable = true)
 |    |-- 2. Symbol: string (nullable = true)
 |    |-- 3. Last Refreshed: string (nullable = true)
 |    |-- 4. Output Size: string (nullable = true)
 |    |-- 5. Time Zone: string (nullable = true)
 |-- Time Series (Daily): struct (nullable = true)
 |    |-- 2025-11-28: struct (nullable = true)
 |    |    |-- 1. open: string (nullable = true)
 |    |    |-- 2. high: string (nullable = true)
 |    |    |-- 3. low: string (nullable = true)
 |    |    |-- 4. close: string (nullable = true)
 |    |    |-- 5. volume: string (nullable = true)
 |    |-- 2025-12-01: struct (nullable = true)
 |    |    |-- 1. open: string (nullable = true)
 |    |    |-- 2. high: string (nullable = true)
 |    |    |-- 3. low: string (nullable = true)
 |    |    |-- 4. close: string (nullable = true)
 |    |    |-- 5. volume: string (nullable = true)
 |    |-- 2025-12-02: struct (nullable = true)
 |    |    |-- 1.

## Step 3 - Flatten Nested JSON

Alpha Vantage stores daily prices inside nested JSON.

We convert it into a proper table format:

symbol | date | open | high | low | close | volume

In [0]:
from pyspark.sql.functions import col, explode, create_map, lit
from itertools import chain

# Get all available date fields from Time Series (Daily)
date_fields = df_raw.schema["Time Series (Daily)"].dataType.fieldNames()

# Convert date columns into map entries
mapping_expr = create_map(
    list(chain.from_iterable(
        [(lit(date), col(f"`Time Series (Daily)`.`{date}`")) for date in date_fields]
    ))
)

df_flat = (
    df_raw
    .select(
        col("`Meta Data`.`2. Symbol`").alias("symbol"),
        explode(mapping_expr).alias("date", "values")
    )
    .select(
        col("symbol"),
        col("date"),
        col("values.`1. open`").cast("double").alias("open"),
        col("values.`2. high`").cast("double").alias("high"),
        col("values.`3. low`").cast("double").alias("low"),
        col("values.`4. close`").cast("double").alias("close"),
        col("values.`5. volume`").cast("long").alias("volume")
    )
)

display(df_flat)

symbol,date,open,high,low,close,volume
IBM,2025-11-28,304.06,309.18,303.6,308.58,1689031
IBM,2025-12-01,306.505,307.12,302.8,305.67,3166555
IBM,2025-12-02,307.0,310.4675,301.57,301.78,4261100
IBM,2025-12-03,302.88,303.97,298.905,302.62,3953390
IBM,2025-12-04,302.875,309.61,302.54,307.99,2962463
IBM,2025-12-05,308.59,311.83,307.18,307.94,2344667
IBM,2025-12-08,309.62,315.3454,307.95,309.18,3615794
IBM,2025-12-09,309.63,313.97,308.75,310.48,2914275
IBM,2025-12-10,310.23,314.69,306.6513,312.67,3411524
IBM,2025-12-11,312.0,313.44,308.4,310.74,2755749


## Step 4 - Define DLT Tables

Now that the flattening logic works, we convert it into Delta Live Tables.

DLT will create:
- Bronze table from raw JSON
- Silver table from flattened clean data
- Gold tables from aggregated business metrics

In [0]:
import dlt
from pyspark.sql.functions import (
    col, explode, create_map, lit, current_timestamp,
    avg, max, min, sum, count, round
)
from itertools import chain

raw_path = "/Volumes/databricks_workspace/stock_dlt/raw_data"


@dlt.table(
    name="bronze_stock_raw",
    comment="Raw Alpha Vantage JSON files"
)
def bronze_stock_raw():
    return spark.read.json(raw_path)


@dlt.table(
    name="silver_stock_prices",
    comment="Flattened and cleaned daily stock prices"
)
def silver_stock_prices():
    df_raw = dlt.read("bronze_stock_raw")

    date_fields = df_raw.schema["Time Series (Daily)"].dataType.fieldNames()

    mapping_expr = create_map(
        list(chain.from_iterable(
            [(lit(date), col(f"`Time Series (Daily)`.`{date}`")) for date in date_fields]
        ))
    )

    return (
        df_raw
        .select(
            col("`Meta Data`.`2. Symbol`").alias("symbol"),
            explode(mapping_expr).alias("date", "values")
        )
        .select(
            col("symbol"),
            col("date").cast("date").alias("trade_date"),
            col("values.`1. open`").cast("double").alias("open"),
            col("values.`2. high`").cast("double").alias("high"),
            col("values.`3. low`").cast("double").alias("low"),
            col("values.`4. close`").cast("double").alias("close"),
            col("values.`5. volume`").cast("long").alias("volume"),
            current_timestamp().alias("processed_at")
        )
    )


@dlt.table(
    name="gold_stock_summary",
    comment="Summary metrics by stock symbol"
)
def gold_stock_summary():
    df = dlt.read("silver_stock_prices")

    return (
        df.groupBy("symbol")
        .agg(
            round(avg("close"), 2).alias("avg_close_price"),
            round(max("close"), 2).alias("max_close_price"),
            round(min("close"), 2).alias("min_close_price"),
            sum("volume").alias("total_volume"),
            count("*").alias("trading_days")
        )
    )


@dlt.table(
    name="gold_daily_price_trend",
    comment="Daily close price and volume trend"
)
def gold_daily_price_trend():
    return (
        dlt.read("silver_stock_prices")
        .select("symbol", "trade_date", "close", "volume")
    )